In [38]:
'''
Author: David Brin
Date Rewritten: 1/23/2025

This notebook is a repurposed notebook to recreate the data frame containing all the spikes and their features 
along with the dictionary containing all the spike waveforms that will correspond to the spikes in the data frame via spike ID. 
'''

'\nAuthor: David Brin\nDate Rewritten: 1/23/2025\n\nThis notebook is a repurposed notebook to recreate the data frame containing all the spikes and their features \nalong with the dictionary containing all the spike waveforms that will correspond to the spikes in the data frame via spike ID. \n'

In [39]:
#imports
%matplotlib inline
%config InlineBackend.figure_format = 'retina' # high res plotting

import sys
sys.path.append(r'..\..\..\spikeparam')

from spikeparam.patch.fit import Spike
from spikeparam.patch.fit import SpikeGroup

from neurodsp import spectral

from scipy import signal
import scipy

import h5py
from tqdm import tqdm

import numpy as np
import pandas as pd

from neurodsp import filt
from neurodsp.timefrequency import amp_by_time, phase_by_time
from neurodsp.plts import plot_time_series, plot_instantaneous_measure
from neurodsp.plts.time_series import plot_bursts
from neurodsp.burst import detect_bursts_dual_threshold, compute_burst_stats

from scipy.signal import sosfiltfilt, butter

from scipy.signal import find_peaks
from scipy.optimize import curve_fit
from scipy.stats import pearsonr
import statsmodels.api as sm
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import scipy.spatial as sp_spatial

import os

from fooof import FOOOF

sns.set(rc={'figure.figsize':(12,9)})
sns.set_style('whitegrid')
sns.set_style("whitegrid", {'axes.grid' : False})

import IProgress

import openpyxl
import pickle

## Defining useful functions

In [40]:
# Function to visualize sweep patch data
# no metadata, just the time series data
def extract_data(file_path, plot_data = False):
    # Open the HDF5 file
    with h5py.File(file_path, 'r') as f:
        # Initialize an empty list to store data arrays
        data = []

        # Iterate through keys in the 'acquisition' group
        for sweep_key in f['acquisition'].keys():
            dataset = f['acquisition'][sweep_key]['data'] 
            # Convert the dataset data into a NumPy array and append to the list
            data.append(np.array(dataset))

        # Plot the data
        if(plot_data):
            if all(d.ndim == 1 for d in data):
                for d in data:
                    plt.plot(d)
                plt.xlabel('time (ms)')
                plt.ylabel('mV')
                plt.title('1D Dataset Visualization')
                plt.show()
            elif all(d.ndim == 2 for d in data):
                for d in data:
                    plt.imshow(d, cmap='viridis')
                    plt.colorbar()
                    plt.xlabel('X-axis')
                    plt.ylabel('Y-axis')
                    plt.title('2D Dataset Visualization')
                    plt.show()
            else:
                print("Cannot visualize data with more than 2 dimensions.")



        return data

In [41]:
#collecting file paths

def get_file_paths(folder_path):
    """
    Function to loop through a folder and save file paths.
    
    Args:
    - folder_path (str): Path to the folder to loop through.
    
    Returns:
    - file_paths (list): List of file paths found in the folder.
    """
    file_paths = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_paths.append(os.path.join(root, file))
    return file_paths 


In [42]:
def update_columns_at_index(df, file_path):
    """
    Function to update columns in the DataFrame at a specific index with values from Excel metadata.
    
    Check if filename matches the string in the first cell of the row, drop row if not.

    Args:
    - df (pd.DataFrame): DataFrame to update.
    - file_path (str): Path to the Excel file containing metadata.
    
    Returns:
    - true if updated
    """
    # Extract filename from file_path
    filename = os.path.splitext(os.path.basename(file_path))[0]
    
    # Load the workbook
        #wb = openpyxl.load_workbook(r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\ephys_features_filenames (1).xlsx")
    metadf = pd.read_excel(r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\ephys_features_filenames (1).xlsx")
        # Select the active worksheet
        #ws = wb.active

    

    # Filter the metadata DataFrame to find the row associated with this file
    file_metadata = metadf[metadf['file_name'] == filename]
    #print(file_metadata.head())
    if not file_metadata.empty:    
        for col in file_metadata.columns:
            #print(col)
            df[col] = file_metadata[col].values[0]  # Assign metadata value to the entire column
 
        return True
    
    else:
        print(f"No metadata found for file: {filename}")
        return False


In [43]:
def monkey_df_and_dict(filepaths):
    """
    Creates a DataFrame and dictionary of spike data for all files in the given filepaths (Monkey Directoru.

    Parameters:
    - filepaths: List of file paths to process.
    - ind_start: Starting index for the metadata cols for 'update_columns_at_index' for the monkey data frame.

    Returns:
    - super_mega_df: A concatenated DataFrame of spike features for all files.
    """
    super_mega_df = pd.DataFrame()
    monkey_dict = {}  # Monkey dictionary
    file_num = 0

    for file in filepaths:
        print(file)
        # Extract data for all sweeps in the file
        data = extract_data(file, plot_data=False)  # Numpy array of all sweeps in the file
        spike_dir = {}  # Dictionary of sweeps with spikes
        i = 0
        monkey_id = os.path.basename(os.path.dirname(file))  # Extract Monkey_ID from the directory name
        fileName = os.path.splitext(os.path.basename(file))[0] 
        with h5py.File(file, 'r') as f:
            for sweep_key in f['acquisition'].keys():
                if i < len(data):
                    try:
                        # Fit spike data
                        sweep_key_obj = Spike(thresh_amp=0, window_length=(5., 5.), smooth_frac=.01)
                        sweep_key_obj.fit(data[i], 20000, n_jobs=-1, progress=tqdm)
                        if sweep_key_obj.n_spikes is not None:
                            spike_dir[sweep_key] = sweep_key_obj
                            #print(f'num spikes: {sweep_key_obj.n_spikes} and len df: {len(sweep_key_obj.df_features)}')
                            #print(f"Length of sweep_key_obj.spikes: {len(sweep_key_obj.spikes)}")
                    except ValueError as e:
                        print(f"Fitting failed for sweep {sweep_key}: {e}")
                i += 1

        # Create and concatenate DataFrame with all sweeps
        mega_df = pd.DataFrame()
        mega_dict = {}             #store all spikes in file and concatenate if file appears in metadata spreadsheet
        for i, (sweep_key, sweep_obj) in enumerate(spike_dir.items()):
            print(sweep_key, sweep_obj)
            df = sweep_obj.df_features.copy()  # Copy DataFrame to avoid modifying the original
            
            df['Sweep_#'] = sweep_key
            df['Spike_#'] = range(0, len(df))  # Add Spike_# as a column
            
            # Create Spike_IDs in the desired format
            df['Spike_ID'] = df.apply(
                lambda row: f"{monkey_id}f{file_num}Sw{sweep_key}Sp{row['Spike_#']}", axis=1
            )
            #display(df)
            for spike in range(len(df)):
                key = f"{monkey_id}f{file_num}Sw{sweep_key}Sp{spike}"                      
                mega_dict[key] = sweep_obj.spikes[spike]
            # Concatenate the current DataFrame to the mega DataFrame
            
            mega_df = pd.concat([mega_df, df], axis=0)
            #print(f'num spikes: {sweep_obj.n_spikes} and len df: {len(sweep_obj.df_features)}')

        # Update columns and append to the super_mega_df
        exists = update_columns_at_index(mega_df, file)
        if exists:
            super_mega_df = pd.concat([super_mega_df, mega_df], axis=0)
            monkey_dict.update(mega_dict)
            print("File added")
            print(f"\n\n\n\ndf length: {len(super_mega_df)} \n dict length: {len(monkey_dict.keys())} \n\n\n\n")
        
        file_num += 1

    return super_mega_df, monkey_dict


In [44]:
def create_allMonkey_data(file_paths):
    '''
    cohesive function that calls monkey_df_and_dict on all monkey folders, puts all data into allMonkey_df and combined_dict
    couldn't make it fully adaptable because of 'update_columns_at_index'
    
    returns allMonkey_df, combined_dict
    '''
    allMonkey_df = pd.DataFrame()
    combined_dict = {}
    print("Creating data frame and dictionary")
    for file in file_paths:
        file_list = [os.path.join(file, f) for f in os.listdir(file) if os.path.isfile(os.path.join(file, f))]
        monk_df,monk_dict = monkey_df_and_dict(file_list)     
        if not monk_df.empty:
            allMonkey_df = pd.concat([allMonkey_df, monk_df], axis=0, ignore_index=True)
        else:
            print(f"\n\n\n\n\nWarning: No data found in {file}\n\n\n\n\n\n\n\n\n") #run a check in case empty file
        combined_dict.update(monk_dict)
        print(f"\n\n\n\n\n\n\nconcat df length: {len(allMonkey_df)} \nconcat dict length: {len(combined_dict.keys())} \n\n\n\n\n\n\n")
    return allMonkey_df, combined_dict
    

In [45]:
# Functions to save a datrame to a pickle file and another to extract the data from the pickle file

def save_dataframe_to_pickle(dataframe, file_path):
    """
    Function to save a DataFrame as a pickle file.
    
    Args:
    - dataframe (pd.DataFrame): DataFrame to be saved.
    - file_path (str): Path to save the pickle file.
    """
    dataframe.to_pickle(file_path)
    print(f"Data frame saved to {file_path}")

def load_dataframe_from_pickle(file_path):
    """
    Function to extract a DataFrame from a pickle file.
    
    Args:
    - file_path (str): Path to the pickle file.
    
    Returns:
    - dataframe (pd.DataFrame): Loaded DataFrame.
    """
    dataframe = pd.read_pickle(file_path)
    return dataframe

def save_dict_to_pickle(dictionary, filepath):
    """
    Save a dictionary to a pickle file.

    Parameters:
        dictionary (dict): The dictionary to save.
        filepath (str): The path to the pickle file.
    """
    with open(filepath, 'wb') as f:
        pickle.dump(dictionary, f)
    print(f"Dictionary saved to {filepath}")


def load_dict_from_pickle(filepath):
    """
    Load a dictionary from a pickle file.

    Parameters:
        filepath (str): The path to the pickle file.

    Returns:
        dict: The loaded dictionary.
    """
    with open(filepath, 'rb') as f:
        dictionary = pickle.load(f)
    print(f"Dictionary loaded from {filepath}")
    return dictionary

## Extracting Data

In [46]:
file_paths = [r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M03", r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M04", r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M05"
             ,r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M06", r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M08"
             , r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M10", r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M11"
             , r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M12", r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M19"
             , r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M20", r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M21"]

allMonkey_df, combined_dict = create_allMonkey_data(file_paths)
save_dataframe_to_pickle(allMonkey_df, r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\allMonkey_df.pkl")
save_dict_to_pickle(combined_dict, r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\combined_dict.pkl")

Creating data frame and dictionary
C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M03\M03_JS_A1_C01.nwb


C:\Users\david\Documents\Voytek Research\spikeparam\datasets\primate_dataset\..\..\..\spikeparam\spikeparam\patch\fit\fit.py:242: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 15/15 [00:04<00:00,  3.01it/s]
C:\Users\david\Documents\Voytek Research\spikeparam\datasets\primate_dataset\..\..\..\spikeparam\spikeparam\patch\fit\fit.py:242: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.85s/it]
C:\Users\david\Documents\Voytek Research\spikeparam\datasets\primate_dataset\..\..\..\spikeparam\spikeparam\patch\fit\fit.py:242: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.11it/s]


Sweep_10 <spikeparam.patch.fit.fit.Spike object at 0x0000014C53F9B2C0>
Sweep_11 <spikeparam.patch.fit.fit.Spike object at 0x0000014C537D22D0>
Sweep_12 <spikeparam.patch.fit.fit.Spike object at 0x0000014C53A0DDF0>
Sweep_13 <spikeparam.patch.fit.fit.Spike object at 0x0000014C58953200>
Sweep_14 <spikeparam.patch.fit.fit.Spike object at 0x0000014C58952BD0>
Sweep_54 <spikeparam.patch.fit.fit.Spike object at 0x0000014C5313C680>
Sweep_7 <spikeparam.patch.fit.fit.Spike object at 0x0000014C5313C830>
Sweep_8 <spikeparam.patch.fit.fit.Spike object at 0x0000014C59F474A0>
Sweep_9 <spikeparam.patch.fit.fit.Spike object at 0x0000014C531721B0>
File added




df length: 68 
 dict length: 68 




C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M03\M03_JS_A1_C03.nwb


C:\Users\david\Documents\Voytek Research\spikeparam\datasets\primate_dataset\..\..\..\spikeparam\spikeparam\patch\fit\fit.py:242: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 16/16 [00:04<00:00,  3.33it/s]
C:\Users\david\Documents\Voytek Research\spikeparam\datasets\primate_dataset\..\..\..\spikeparam\spikeparam\patch\fit\fit.py:242: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.88s/it]
C:\Users\david\Documents\Voytek Research\spikeparam\datasets\primate_dataset\..\..\..\spikeparam\spikeparam\patch\fit\fit.py:242: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.88s/it]
C:\Users\david\Documents\Voytek Re

Sweep_10 <spikeparam.patch.fit.fit.Spike object at 0x0000014C53F836B0>
Sweep_11 <spikeparam.patch.fit.fit.Spike object at 0x0000014C58F4B530>
Sweep_12 <spikeparam.patch.fit.fit.Spike object at 0x0000014C53F821E0>
Sweep_13 <spikeparam.patch.fit.fit.Spike object at 0x0000014C5B107D10>
Sweep_14 <spikeparam.patch.fit.fit.Spike object at 0x0000014C560A6000>
Sweep_38 <spikeparam.patch.fit.fit.Spike object at 0x0000014C557BD790>
Sweep_39 <spikeparam.patch.fit.fit.Spike object at 0x0000014C557BE900>
Sweep_40 <spikeparam.patch.fit.fit.Spike object at 0x0000014C5380CE30>
File added




df length: 143 
 dict length: 143 




C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M03\M03_JS_A1_C04.nwb


...

Sweep_4 <spikeparam.patch.fit.fit.Spike object at 0x0000014C5B523B60>
Sweep_5 <spikeparam.patch.fit.fit.Spike object at 0x0000014C59F494C0>
Sweep_6 <spikeparam.patch.fit.fit.Spike object at 0x0000014C5AFBCD10>
Sweep_7 <spikeparam.patch.fit.fit.Spike object at 0x0000014C5617EFC0>
Sweep_8 <spikeparam.patch.fit.fit.Spike object at 0x0000014C58956450>
Sweep_9 <spikeparam.patch.fit.fit.Spike object at 0x0000014C58931A90>
File added




df length: 1273 
 dict length: 1273 




C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M21\M21_SA_A1_C06.nwb


C:\Users\david\Documents\Voytek Research\spikeparam\datasets\primate_dataset\..\..\..\spikeparam\spikeparam\patch\fit\fit.py:242: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 38/38 [00:04<00:00,  8.55it/s]
C:\Users\david\Documents\Voytek Research\spikeparam\datasets\primate_dataset\..\..\..\spikeparam\spikeparam\patch\fit\fit.py:242: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 6/6 [00:02<00:00,  2.29it/s]


Sweep_10 <spikeparam.patch.fit.fit.Spike object at 0x0000014C552C7260>
Sweep_11 <spikeparam.patch.fit.fit.Spike object at 0x0000014C5B2E42C0>
Sweep_12 <spikeparam.patch.fit.fit.Spike object at 0x0000014C53F80410>
Sweep_8 <spikeparam.patch.fit.fit.Spike object at 0x0000014C5895F560>
Sweep_9 <spikeparam.patch.fit.fit.Spike object at 0x0000014C5634D730>
File added




df length: 1380 
 dict length: 1380 




C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\M21\M21_SA_A1_C07.nwb


C:\Users\david\Documents\Voytek Research\spikeparam\datasets\primate_dataset\..\..\..\spikeparam\spikeparam\patch\fit\fit.py:242: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 25/25 [00:04<00:00,  5.68it/s]
C:\Users\david\Documents\Voytek Research\spikeparam\datasets\primate_dataset\..\..\..\spikeparam\spikeparam\patch\fit\fit.py:242: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.98it/s]


Sweep_10 <spikeparam.patch.fit.fit.Spike object at 0x0000014C53F83350>
Sweep_11 <spikeparam.patch.fit.fit.Spike object at 0x0000014C5602C3B0>
Sweep_12 <spikeparam.patch.fit.fit.Spike object at 0x0000014C5626D370>
Sweep_13 <spikeparam.patch.fit.fit.Spike object at 0x0000014C59F4AA80>
Sweep_8 <spikeparam.patch.fit.fit.Spike object at 0x0000014C557BE090>
Sweep_9 <spikeparam.patch.fit.fit.Spike object at 0x0000014C53F80410>
File added




df length: 1463 
 dict length: 1463 











concat df length: 12550 
concat dict length: 12550 







Data frame saved to C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\allMonkey_df.pkl
Dictionary saved to C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\combined_dict.pkl


In [47]:
allMonkey_df.head()

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,...,Spike_#,Spike_ID,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species
0,2.612488,0.45,-45.858766,15.164185,0.50,6.445313,6.970228,-56.733916,88.40,0.996142,...,0,M03f0SwSweep_10Sp0,M03_JS_A1_C01,A,3.0,PFC,6.3,F,9.96,Macaca fascicularis
1,1.832248,0.60,-38.827516,11.984253,0.55,3.929138,5.981717,-53.646425,56.35,0.950116,...,1,M03f0SwSweep_10Sp1,M03_JS_A1_C01,A,3.0,PFC,6.3,F,9.96,Macaca fascicularis
2,1.725643,0.60,-38.571168,11.453247,0.55,3.533936,6.194347,-53.275150,56.20,0.969743,...,2,M03f0SwSweep_10Sp2,M03_JS_A1_C01,A,3.0,PFC,6.3,F,9.96,Macaca fascicularis
3,1.762723,0.60,-39.727784,11.669922,0.60,3.567505,5.819483,-53.277108,NaN,0.975530,...,3,M03f0SwSweep_10Sp3,M03_JS_A1_C01,A,3.0,PFC,6.3,F,9.96,Macaca fascicularis
4,2.534657,0.50,-46.746827,15.621949,0.50,6.254578,7.821646,-56.351455,90.70,0.982983,...,0,M03f0SwSweep_11Sp0,M03_JS_A1_C01,A,3.0,PFC,6.3,F,9.96,Macaca fascicularis


In [31]:
save_dataframe_to_pickle(allMonkey_df, r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\allMonkey_df.pkl")

Data frame saved to C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\allMonkey_df.pkl


**OLD CODE:**

In [14]:
file_path = [r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\Test"]
test_df, test_dict = create_allMonkey_data(file_path)
test_df.head()

Creating data frame and dictionary
C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\Test\M05_JS_A1_C03.nwb


C:\Users\david\Documents\Voytek Research\spikeparam\datasets\primate_dataset\..\..\..\spikeparam\spikeparam\patch\fit\fit.py:242: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 16/16 [00:07<00:00,  2.27it/s]
C:\Users\david\Documents\Voytek Research\spikeparam\datasets\primate_dataset\..\..\..\spikeparam\spikeparam\patch\fit\fit.py:242: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 14/14 [00:06<00:00,  2.27it/s]
C:\Users\david\Documents\Voytek Research\spikeparam\datasets\primate_dataset\..\..\..\spikeparam\spikeparam\patch\fit\fit.py:242: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 15/15 [00:06<00:00,  2.37it/s]


Sweep_10 <spikeparam.patch.fit.fit.Spike object at 0x0000014C530AD670>
Sweep_11 <spikeparam.patch.fit.fit.Spike object at 0x0000014C51D06180>
Sweep_12 <spikeparam.patch.fit.fit.Spike object at 0x0000014C53053DD0>
Sweep_13 <spikeparam.patch.fit.fit.Spike object at 0x0000014C52F79F70>
Sweep_14 <spikeparam.patch.fit.fit.Spike object at 0x0000014C5302A300>
Sweep_22 <spikeparam.patch.fit.fit.Spike object at 0x0000014C2DFC2390>
Sweep_23 <spikeparam.patch.fit.fit.Spike object at 0x0000014C530E0230>
Sweep_24 <spikeparam.patch.fit.fit.Spike object at 0x0000014C516DA3C0>
Sweep_25 <spikeparam.patch.fit.fit.Spike object at 0x0000014C52E735F0>
Sweep_26 <spikeparam.patch.fit.fit.Spike object at 0x0000014C0F12D8B0>
Sweep_27 <spikeparam.patch.fit.fit.Spike object at 0x0000014C53052030>
Sweep_28 <spikeparam.patch.fit.fit.Spike object at 0x0000014C52F8A630>
Sweep_29 <spikeparam.patch.fit.fit.Spike object at 0x0000014C52F8B4D0>
Sweep_7 <spikeparam.patch.fit.fit.Spike object at 0x0000014C5313C350>
Sweep_8

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,r_squared_exp,Sweep_#,Spike_#,Spike_ID,file_name
0,3.007244,0.55,-42.572022,35.583497,0.85,4.592896,2.412379,-51.655136,80.95,0.953042,0.992757,Sweep_10,0,Testf0SwSweep_10Sp0,M05_JS_A1_C03
1,2.759432,0.70,-37.170411,33.142091,1.00,2.136231,1.831402,-52.007353,68.75,0.952943,0.996262,Sweep_10,1,Testf0SwSweep_10Sp1,M05_JS_A1_C03
2,2.428557,0.70,-36.895753,32.653809,1.00,1.403809,1.822783,-51.590403,77.10,0.909808,0.997062,Sweep_10,2,Testf0SwSweep_10Sp2,M05_JS_A1_C03
3,2.990264,0.75,-36.132813,32.287598,1.00,1.434326,1.759038,-52.521420,78.40,0.953441,0.997450,Sweep_10,3,Testf0SwSweep_10Sp3,M05_JS_A1_C03
4,3.344084,0.70,-36.102296,32.135010,0.95,1.525879,1.818987,-52.357826,85.60,0.955121,0.996327,Sweep_10,4,Testf0SwSweep_10Sp4,M05_JS_A1_C03


In [12]:
file_paths = get_file_paths(r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M03")
M03_df = monkey_df(file_paths, 2)

save_dataframe_to_pickle(M03_df, r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M03_df.pkl")

C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M03\M03_JS_A1_C01.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 67/67 [00:04<00:00, 14.38it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.44s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 5/5 [00:02<00:00,  1.75it/s]


Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_54
Sweep_7
Sweep_8
Sweep_9
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M03\M03_JS_A1_C03.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████| 166/166 [00:04<00:00, 34.97it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.32s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.38s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_38
Sweep_39
Sweep_40
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M03\M03_JS_A1_C04.nwb


...

C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.43s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


Sweep_12
Sweep_13
Sweep_14
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M03\M03_SM_A1_C16.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.30s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.45s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|████████████████████████████████████████████████████████████████████████████| 2/2 [16:00<00:00, 480.17s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.44s/

Sweep_10
Sweep_11
Sweep_115
Sweep_116
Sweep_117
Sweep_118
Sweep_119
Sweep_12
Sweep_120
Sweep_121
Sweep_122
Sweep_123
Sweep_124
Sweep_125
Sweep_126
Sweep_13
Sweep_14
Sweep_8
Sweep_9



C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


In [14]:
M03_df.head()

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,r_squared_exp,Sweep_#,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species
0,2.612488,0.45,-45.858766,15.164185,0.50,6.445313,6.970228,-56.733916,19.45,0.996142,0.739826,Sweep_10,M03_JS_A1_C01,A,3,PFC,6.3,F,9.96,Macaca fascicularis
1,2.116406,0.55,-42.507936,13.681031,0.50,5.018616,7.193608,-54.574427,20.15,0.998421,0.688264,Sweep_10,M03_JS_A1_C01,A,3,PFC,6.3,F,9.96,Macaca fascicularis
2,2.044173,0.60,-41.433717,12.530518,0.55,3.962708,6.693805,-54.071263,24.25,0.984082,0.725384,Sweep_10,M03_JS_A1_C01,A,3,PFC,6.3,F,9.96,Macaca fascicularis
3,1.850421,0.60,-38.183595,11.666870,0.55,3.634644,6.344601,-53.386135,24.55,0.994742,0.774003,Sweep_10,M03_JS_A1_C01,A,3,PFC,6.3,F,9.96,Macaca fascicularis
4,1.832248,0.60,-38.827516,11.984253,0.55,3.929138,5.981717,-53.646425,26.60,0.950116,0.831859,Sweep_10,M03_JS_A1_C01,A,3,PFC,6.3,F,9.96,Macaca fascicularis


In [15]:
M03_df.tail()

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,r_squared_exp,Sweep_#,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species
0,3.532513,0.50,-41.070558,34.521485,0.60,9.402466,5.926023,-41.947774,NaN,0.990231,0.869755,Sweep_13,M03_SM_A1_C16,NA,NA,PFC,6.3,F,9.96,Macaca fascicularis
0,3.611996,0.50,-41.882325,35.430909,0.55,8.175659,7.009688,-41.227819,6.55,0.979050,0.715464,Sweep_14,M03_SM_A1_C16,NA,NA,PFC,6.3,F,9.96,Macaca fascicularis
1,2.420113,1.15,-35.577393,14.031983,0.90,1.815796,3.344942,-40.903731,NaN,0.992774,0.984037,Sweep_14,M03_SM_A1_C16,NA,NA,PFC,6.3,F,9.96,Macaca fascicularis
0,3.556652,0.60,-29.888917,27.313233,0.70,3.521729,3.635624,-45.651009,NaN,0.986556,0.996845,Sweep_8,M03_SM_A1_C16,NA,NA,PFC,6.3,F,9.96,Macaca fascicularis
0,2.805506,0.55,-37.921143,33.331300,0.60,6.616211,4.633209,-44.813706,NaN,0.988634,0.978401,Sweep_9,M03_SM_A1_C16,NA,NA,PFC,6.3,F,9.96,Macaca fascicularis


**M04**

In [60]:
file_paths2 = get_file_paths(r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M04")
M04_df = monkey_df(file_paths2, 28)

save_dataframe_to_pickle(M04_df, r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M04_df.pkl")

C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M04\M04_JS_A1_C01.nwb


Spike: 100%|█████████████████████████████████████████████████████████████████████████| 104/104 [00:03<00:00, 30.74it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
file added
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M04\M04_JS_A1_C02.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 30/30 [00:03<00:00,  8.32it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.99it/s]


...

Spike: 100%|███████████████████████████████████████████████████████████████████████████| 88/88 [00:03<00:00, 23.80it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.19s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 43/43 [00:03<00:00, 11.88it/s]


Sweep_0
Sweep_1
Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_18
Sweep_19
Sweep_2
Sweep_20
Sweep_21
Sweep_24
Sweep_25
Sweep_26
Sweep_27
Sweep_28
Sweep_29
Sweep_3
Sweep_30
Sweep_31
Sweep_32
Sweep_33
Sweep_34
Sweep_4
Sweep_5
Sweep_6
Sweep_7
Sweep_8
Sweep_9
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M04\M04_SM_A1_C13.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.59s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.56s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.29s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.55s/

Sweep_13
Sweep_14
Sweep_200
Sweep_205
Sweep_206
file added



C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


In [61]:
M04_df.head()

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,r_squared_exp,Sweep_#,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species
0,4.387923,0.45,-40.768434,24.624634,0.40,9.146118,7.228195,-61.191912,3141.0,0.988773,0.670360,Sweep_10,M04_JS_A1_C01,NA,NA,PFC,6.5,F,11.58,Macaca fascicularis
1,4.833480,0.40,-41.177369,25.955201,0.40,10.462952,8.280124,-59.228859,NaN,0.990291,0.534707,Sweep_10,M04_JS_A1_C01,NA,NA,PFC,6.5,F,11.58,Macaca fascicularis
0,3.545225,0.40,-40.933228,24.578858,0.40,10.218811,7.681916,-60.576132,13.8,0.985757,0.618962,Sweep_11,M04_JS_A1_C01,NA,NA,PFC,6.5,F,11.58,Macaca fascicularis
1,3.574366,0.45,-38.961793,22.283936,0.45,8.531189,6.139406,-60.119660,15.2,0.983322,0.773941,Sweep_11,M04_JS_A1_C01,NA,NA,PFC,6.5,F,11.58,Macaca fascicularis
2,4.571855,0.45,-35.510255,20.413208,0.40,6.814575,5.891516,-59.775511,13.5,0.987037,0.753234,Sweep_11,M04_JS_A1_C01,NA,NA,PFC,6.5,F,11.58,Macaca fascicularis


In [62]:
M04_df = load_dataframe_from_pickle(r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M04_df.pkl")

M04_df.tail()

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,r_squared_exp,Sweep_#,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species
0,0.672305,0.80,-41.328373,41.478031,0.95,2.859497,2.601819,-48.453468,NaN,0.865878,0.998288,Sweep_13,M04_SM_A1_C13,NA,NA,PFC,6.5,F,11.58,Macaca fascicularis
0,0.870738,0.80,-42.854252,41.923588,0.90,3.094482,2.641498,-48.066803,NaN,0.978995,0.998324,Sweep_14,M04_SM_A1_C13,NA,NA,PFC,6.5,F,11.58,Macaca fascicularis
0,6.746175,0.75,-47.212162,41.520756,0.85,3.582764,2.447608,-56.480353,NaN,0.998580,0.991162,Sweep_200,M04_SM_A1_C13,NA,NA,PFC,6.5,F,11.58,Macaca fascicularis
0,7.036114,0.75,-46.192875,42.369144,0.85,3.350830,2.516052,-55.528810,NaN,0.998975,0.991049,Sweep_205,M04_SM_A1_C13,NA,NA,PFC,6.5,F,11.58,Macaca fascicularis
0,7.153687,0.75,-46.510258,42.546146,0.85,3.405762,2.494742,-55.589845,NaN,0.999300,0.990622,Sweep_206,M04_SM_A1_C13,NA,NA,PFC,6.5,F,11.58,Macaca fascicularis


In [63]:
file_paths3 = get_file_paths(r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M05")
M05_df = monkey_df(file_paths3, 49)

save_dataframe_to_pickle(M05_df, r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M05_df.pkl")

C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M05\M05_JS_A1_C01.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  2.15it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.18it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.20it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.18s/

Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_15
Sweep_16
Sweep_17
Sweep_18
Sweep_19
Sweep_2
Sweep_20
Sweep_21
Sweep_22
Sweep_23
Sweep_24
Sweep_25
Sweep_26
Sweep_27
Sweep_28
Sweep_29
Sweep_3
Sweep_4
Sweep_5
Sweep_53
Sweep_55
Sweep_56
Sweep_57
Sweep_58
Sweep_59
Sweep_6
Sweep_60
Sweep_61
Sweep_62
Sweep_63
Sweep_64
Sweep_65
Sweep_66
Sweep_67
Sweep_68
Sweep_69
Sweep_7
Sweep_70
Sweep_71
Sweep_8
Sweep_9
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M05\M05_JS_A1_C02.nwb


Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.33s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.52s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 25/25 [00:03<00:00,  7.20it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 18/18 [00:03<00:00,  5.35it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected

Sweep_0
Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_18
Sweep_19
Sweep_20
Sweep_21
Sweep_22
Sweep_23
Sweep_24
Sweep_25
Sweep_26
Sweep_27
Sweep_28
Sweep_29
Sweep_3
Sweep_4
Sweep_5
Sweep_6
Sweep_66
Sweep_67
Sweep_68
Sweep_69
Sweep_7
Sweep_70
Sweep_71
Sweep_72
Sweep_73
Sweep_74
Sweep_75
Sweep_8
Sweep_9
file added
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M05\M05_JS_A1_C03.nwb


...

C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 3/3 [00:01<00:00,  1.68it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.78s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_9
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M05\M05_SM_A1_C13.nwb


Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.86s/it]

Sweep_27
Sweep_28
Sweep_29



C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


In [64]:
M05_df = load_dataframe_from_pickle(r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M05_df.pkl")

M05_df.tail()

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,r_squared_exp,Sweep_#,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species
4,1.787917,1.20,-35.691650,26.594727,1.45,0.610352,1.165553,-47.850570,43.70,0.908361,0.997375,Sweep_9,M05_JS_A1_C13,NA,NA,PFC,7.4,M,4.39,Macaca fascicularis
5,1.474022,1.15,-35.691650,27.632324,1.40,0.946045,1.132225,-48.452109,51.80,0.890761,0.998009,Sweep_9,M05_JS_A1_C13,NA,NA,PFC,7.4,M,4.39,Macaca fascicularis
6,0.966466,1.20,-35.416992,25.740234,1.50,0.549316,1.122565,-47.547140,0.00,0.796952,0.997495,Sweep_9,M05_JS_A1_C13,NA,NA,PFC,7.4,M,4.39,Macaca fascicularis
7,0.966466,1.20,-35.416992,25.740234,1.50,0.549316,1.122565,-47.547140,52.15,0.796952,0.997495,Sweep_9,M05_JS_A1_C13,NA,NA,PFC,7.4,M,4.39,Macaca fascicularis
8,1.377192,1.30,-34.135254,24.824707,1.50,0.915527,1.099463,-47.206315,NaN,0.774453,0.998393,Sweep_9,M05_JS_A1_C13,NA,NA,PFC,7.4,M,4.39,Macaca fascicularis


In [22]:
file_paths4 = get_file_paths(r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M06")
M06_df = monkey_df(file_paths4, 66)

save_dataframe_to_pickle(M06_df, r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M06_df.pkl")

C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M06\M06_MW_A1_C04.nwb


Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.92s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.79s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.86s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.04s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected

Sweep_0
Sweep_1
Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_15
Sweep_16
Sweep_17
Sweep_2
Sweep_22
Sweep_23
Sweep_24
Sweep_25
Sweep_26
Sweep_27
Sweep_28
Sweep_29
Sweep_3
Sweep_4
Sweep_48
Sweep_49
Sweep_50
Sweep_58
Sweep_60
Sweep_66
Sweep_67
Sweep_68
Sweep_69
Sweep_7
Sweep_70
Sweep_71
Sweep_72
Sweep_73
Sweep_74
Sweep_75
Sweep_76
Sweep_77
Sweep_78
Sweep_79
Sweep_8
Sweep_80
Sweep_81
Sweep_82
Sweep_83
Sweep_84
Sweep_85
Sweep_86
Sweep_87
Sweep_88
Sweep_89
Sweep_9
Sweep_90
Sweep_91
Sweep_92
Sweep_93
Sweep_94
Sweep_95
Sweep_96
Sweep_97
Sweep_98
Sweep_99
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M06\M06_SM_A1_C01.nwb


Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.03s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 5/5 [00:02<00:00,  2.42it/s]


Sweep_0
Sweep_1
Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_2
Sweep_7
Sweep_8
Sweep_9
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M06\M06_SM_A1_C07.nwb


...

Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_21
Sweep_22
Sweep_23
Sweep_24
Sweep_25
Sweep_26
Sweep_27
Sweep_28
Sweep_29
Sweep_6
Sweep_7
Sweep_8
Sweep_9
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M06\M06_SM_A1_C08.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.86s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.92s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


Sweep_39
Sweep_40
Sweep_41
Sweep_42
Sweep_43
Sweep_44
Sweep_45
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M06\M06_SM_A1_C12.nwb


Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 7/7 [00:02<00:00,  2.83it/s]
C:\Python312\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Python312\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 9/9 [00:02<00:00,  3.32it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 8/8 [00:02<00:00,  3.02it/s]


Sweep_0
Sweep_1
Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_2
Sweep_3
Sweep_4
Sweep_7
Sweep_8
Sweep_9


In [23]:
M06_df.head()

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,r_squared_exp,Sweep_#,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species
0,3.015504,0.55,-44.738771,25.451661,0.65,4.821777,2.929958,-52.918481,NaN,0.986424,0.999387,Sweep_0,M06_MW_A1_C04,A,2,PFC,5,M,4.76,Macaca fascicularis
0,2.926016,0.55,-44.158937,26.306153,0.65,4.699707,3.093520,-53.003746,NaN,0.992559,0.998917,Sweep_1,M06_MW_A1_C04,A,2,PFC,5,M,4.76,Macaca fascicularis
0,3.816762,0.50,-41.717530,30.761719,0.60,6.072998,4.063969,-48.254377,14.90,0.994712,0.966464,Sweep_10,M06_MW_A1_C04,A,2,PFC,5,M,4.76,Macaca fascicularis
1,2.270233,0.55,-34.454346,27.465821,0.70,4.013062,2.560024,-48.684115,241.55,0.973479,0.996839,Sweep_10,M06_MW_A1_C04,A,2,PFC,5,M,4.76,Macaca fascicularis
2,1.096338,0.75,-29.998780,28.442383,0.95,1.831055,1.656402,-48.861020,174.35,0.933462,0.998526,Sweep_10,M06_MW_A1_C04,A,2,PFC,5,M,4.76,Macaca fascicularis


In [43]:
M06_df = load_dataframe_from_pickle(r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M06_df.pkl")
M06_df.tail()

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,r_squared_exp,Sweep_#,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species
3,3.509292,0.35,-39.459229,18.096924,0.35,13.931275,10.0,-37.692262,4.60,0.968336,3.119926e-31,Sweep_9,M06_SM_A1_C12,NA,NA,PFC,5,M,4.76,Macaca fascicularis
4,3.587307,0.45,-37.902833,17.181397,0.40,10.894776,10.0,-39.810521,4.95,0.970636,2.524654e-03,Sweep_9,M06_SM_A1_C12,NA,NA,PFC,5,M,4.76,Macaca fascicularis
5,3.275706,0.45,-36.834718,15.563965,0.40,10.421753,10.0,-38.339234,4.85,0.943536,2.168229e-31,Sweep_9,M06_SM_A1_C12,NA,NA,PFC,5,M,4.76,Macaca fascicularis
6,3.401448,0.50,-37.261964,14.373780,0.40,9.506226,10.0,-41.364146,6.50,0.974846,2.094324e-02,Sweep_9,M06_SM_A1_C12,NA,NA,PFC,5,M,4.76,Macaca fascicularis
7,1.979283,0.50,-33.447266,12.359619,0.40,7.080078,10.0,-40.974258,NaN,0.922619,1.694799e-02,Sweep_9,M06_SM_A1_C12,NA,NA,PFC,5,M,4.76,Macaca fascicularis


In [10]:
file_paths5 = get_file_paths(r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M08")
M08_df = monkey_df(file_paths5, 71)

save_dataframe_to_pickle(M06_df, r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M08_df.pkl")

C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M08\M08_JS_A1_C01.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 76/76 [00:04<00:00, 15.24it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 75/75 [00:05<00:00, 14.96it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.56s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_25
Sweep_26
Sweep_27
Sweep_28
Sweep_29
Sweep_84
Sweep_85
Sweep_86
Sweep_87
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M08\M08_JS_A1_C02.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████| 162/162 [00:04<00:00, 33.59it/s]


Fitting failed for sweep Sweep_12: Length of values (163) does not match length of index (162)


...

C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.45s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


Sweep_12
Sweep_13
Sweep_14
Sweep_15
Sweep_16
Sweep_17
Sweep_18
Sweep_19
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M08\M08_SM_A1_C09.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.39it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 7/7 [00:03<00:00,  2.04it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 11/11 [00:03<00:00,  2.88it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 20/20 [00:04<00:00,  4.33it

Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_186
Sweep_187
Sweep_188
Sweep_189
Sweep_190
Sweep_191
Sweep_192
Sweep_193
Sweep_194
Sweep_195
Sweep_196
Sweep_28
Sweep_29
Sweep_30
Sweep_31
Sweep_9


NameError: name 'M06_df' is not defined

In [11]:
M08_df.tail()

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,r_squared_exp,Sweep_#,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species
1,0.794375,0.65,-38.909913,36.468507,0.80,4.425049,2.137163,-51.691492,NaN,0.545088,0.997106,Sweep_30,M08_SM_A1_C09,S,3,PFC,7,F,8.92,Macaca fascicularis
0,1.468515,0.55,-46.234132,40.954591,0.75,6.378174,3.026000,-50.892683,50.2,0.879472,0.991583,Sweep_31,M08_SM_A1_C09,S,3,PFC,7,F,8.92,Macaca fascicularis
1,1.474022,0.65,-38.879395,35.888673,0.80,3.082275,2.198073,-51.345226,NaN,0.874374,0.997618,Sweep_31,M08_SM_A1_C09,S,3,PFC,7,F,8.92,Macaca fascicularis
0,1.390041,0.55,-41.839601,38.360597,0.75,5.432129,2.742456,-52.898929,132.3,0.828544,0.994800,Sweep_9,M08_SM_A1_C09,S,3,PFC,7,F,8.92,Macaca fascicularis
1,1.450618,0.70,-39.520265,33.172608,0.80,3.662109,2.285205,-52.881483,NaN,0.813673,0.997377,Sweep_9,M08_SM_A1_C09,S,3,PFC,7,F,8.92,Macaca fascicularis


In [65]:
file_paths6 = get_file_paths(r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M10")
M10_df = monkey_df(file_paths6, 95)

save_dataframe_to_pickle(M10_df, r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M10_df.pkl")

C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M10\M10_JS_A1_C03.nwb


Spike: 100%|█████████████████████████████████████████████████████████████████████████| 109/109 [00:04<00:00, 24.02it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.08s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 60/60 [00:03<00:00, 16.55it/s]


Sweep_0
Sweep_1
Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_40
Sweep_41
Sweep_42
Sweep_43
Sweep_44
Sweep_45
Sweep_46
Sweep_47
Sweep_48
Sweep_49
Sweep_50
Sweep_51
Sweep_52
Sweep_6
Sweep_7
Sweep_8
Sweep_9
file added
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M10\M10_JS_A1_C04.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  2.04it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.34s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.27s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 60/60 [00:04<00:00, 13.62it

Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_35
Sweep_36
Sweep_37
Sweep_38
Sweep_39
Sweep_40
Sweep_41
Sweep_42
Sweep_43
Sweep_44
Sweep_45
Sweep_46
Sweep_47
Sweep_48
Sweep_49
Sweep_50
Sweep_6
Sweep_7
Sweep_8
Sweep_9
file added
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M10\M10_JS_A1_C05.nwb


...

C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.33s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.29s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.27s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.16it

Sweep_10
Sweep_11
Sweep_115
Sweep_116
Sweep_119
Sweep_12
Sweep_120
Sweep_13
Sweep_14
Sweep_9
file added
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M10\M10_SA_A1_C15.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.24s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.16it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.17it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.26s/

Sweep_101
Sweep_102
Sweep_103
Sweep_104
Sweep_105
Sweep_106
Sweep_107
Sweep_108
Sweep_109
Sweep_110
Sweep_111
Sweep_112
Sweep_113
Sweep_114
Sweep_115
Sweep_116
Sweep_117
Sweep_118
Sweep_119
Sweep_120
Sweep_121
Sweep_122
Sweep_123
Sweep_124
Sweep_125
Sweep_126
Sweep_127
Sweep_128
Sweep_129
Sweep_18
Sweep_97
Sweep_98
file added



C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


In [66]:
M10_df.tail()

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,r_squared_exp,Sweep_#,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species
0,5.624321,0.70,-51.312257,35.900880,1.05,2.520752,1.306941,-55.665414,0.0,0.999225,0.998378,Sweep_129,M10_SA_A1_C15,NA,NA,PFC,5.1,M,5.11,Macaca fascicularis
1,5.624321,0.70,-51.312257,35.900880,1.05,2.520752,1.306941,-55.665414,NaN,0.999225,0.998378,Sweep_129,M10_SA_A1_C15,NA,NA,PFC,5.1,M,5.11,Macaca fascicularis
0,2.568433,0.75,-33.911134,30.499268,1.05,2.270508,0.935556,-49.614184,NaN,0.945924,0.999244,Sweep_18,M10_SA_A1_C15,NA,NA,PFC,5.1,M,5.11,Macaca fascicularis
0,3.764814,0.70,-49.493409,37.994386,1.05,3.094482,1.342215,-54.548994,NaN,0.996289,0.998179,Sweep_97,M10_SA_A1_C15,NA,NA,PFC,5.1,M,5.11,Macaca fascicularis
0,3.466705,0.75,-48.797609,37.640382,1.05,3.399658,1.288728,-54.044722,NaN,0.764764,0.998636,Sweep_98,M10_SA_A1_C15,NA,NA,PFC,5.1,M,5.11,Macaca fascicularis


In [14]:
file_paths7 = get_file_paths(r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M11")
M11_df = monkey_df(file_paths7, 130)

save_dataframe_to_pickle(M11_df, r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M11_df.pkl")

C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M11\M11_SA_A1_C02.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 5/5 [00:03<00:00,  1.66it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.33s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 8/8 [00:03<00:00,  2.38it/s]


Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_42
Sweep_43
Sweep_44
Sweep_45
Sweep_46
Sweep_47
Sweep_7
Sweep_8
Sweep_9


In [15]:
M11_df.tail()

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,r_squared_exp,Sweep_#,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species
3,4.384619,0.65,-40.948487,24.346924,0.85,2.746582,1.752491,-56.577616,41.75,0.947650,0.996799,Sweep_9,M11_SA_A1_C02,S,3,V1,10.6,M,12.69,Macaca mulatta
4,4.359746,0.65,-39.117433,22.375489,0.85,2.416992,1.732443,-56.075213,41.95,0.951499,0.996658,Sweep_9,M11_SA_A1_C02,S,3,V1,10.6,M,12.69,Macaca mulatta
5,4.354973,0.70,-40.454102,22.973633,0.85,2.261353,1.809240,-56.006935,40.95,0.950477,0.996328,Sweep_9,M11_SA_A1_C02,S,3,V1,10.6,M,12.69,Macaca mulatta
6,3.985642,0.70,-40.075684,22.393799,0.85,2.340698,1.760836,-55.809571,47.15,0.960559,0.996670,Sweep_9,M11_SA_A1_C02,S,3,V1,10.6,M,12.69,Macaca mulatta
7,4.839124,0.75,-37.615968,20.031739,0.85,2.011108,1.737839,-55.304615,NaN,0.959954,0.996909,Sweep_9,M11_SA_A1_C02,S,3,V1,10.6,M,12.69,Macaca mulatta


In [67]:
file_paths8 = get_file_paths(r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M12")
M12_df = monkey_df(file_paths8, 131)

save_dataframe_to_pickle(M12_df, r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M12_df.pkl")

C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M12\M12_JS_A1_C01.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████| 144/144 [00:03<00:00, 38.13it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.89s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.98s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████| 147/147 [00:03<00:00, 42.41it

Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_41
Sweep_42
Sweep_43
Sweep_44
Sweep_45
Sweep_46
Sweep_47
Sweep_48
Sweep_49
Sweep_50
Sweep_51
Sweep_52
Sweep_53
Sweep_54
Sweep_55
Sweep_56
Sweep_57
Sweep_58
Sweep_7
Sweep_8
Sweep_9
file added
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M12\M12_JS_A1_C02.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 17/17 [00:03<00:00,  4.99it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 20/20 [00:03<00:00,  5.75it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 21/21 [00:03<00:00,  5.79it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 22/22 [00:03<00:00,  5.84it

Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_154
Sweep_155
Sweep_156
Sweep_6
Sweep_7
Sweep_8
Sweep_9
file added
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M12\M12_JS_A1_C04.nwb


...

C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.16it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


Sweep_21
Sweep_22
Sweep_23
Sweep_24
Sweep_25
Sweep_26
Sweep_27
Sweep_28
file added
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M12\M12_SA_A1_C14.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.19s/it]

Sweep_11
Sweep_12
Sweep_13
Sweep_14
file added



C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


In [68]:
M12_df.tail()

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,r_squared_exp,Sweep_#,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species
1,6.263217,0.55,-38.110111,12.994629,0.55,3.976440,10.000000,-49.644095,NaN,0.997178,0.094598,Sweep_28,M12_SA_A1_C13,NA,NA,LIP,16.2,M,14.38,Macaca mulatta
0,3.142255,0.60,-44.312992,20.915285,0.75,2.908326,3.051178,-53.659981,NaN,0.985559,0.998535,Sweep_11,M12_SA_A1_C14,NA,NA,LIP,16.2,M,14.38,Macaca mulatta
0,4.129373,0.55,-45.716801,21.824709,0.75,3.527832,3.224466,-52.940475,NaN,0.987189,0.997524,Sweep_12,M12_SA_A1_C14,NA,NA,LIP,16.2,M,14.38,Macaca mulatta
0,4.780384,0.50,-49.348392,20.988527,0.70,4.699708,3.410526,-55.296365,NaN,0.991280,0.996018,Sweep_13,M12_SA_A1_C14,NA,NA,LIP,16.2,M,14.38,Macaca mulatta
0,5.296567,0.55,-51.661625,18.608156,0.75,3.240968,3.537176,-55.982380,NaN,0.996950,0.991853,Sweep_14,M12_SA_A1_C14,NA,NA,LIP,16.2,M,14.38,Macaca mulatta


In [71]:
file_paths9 = get_file_paths(r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M19")
M19_df = monkey_df(file_paths9, 147)

save_dataframe_to_pickle(M19_df, r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M19_df.pkl")

C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M19\M19_JS_A1_C01.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.17it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.05s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.34s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.20s/

Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_36
Sweep_39
Sweep_44
Sweep_45
Sweep_46
Sweep_47
Sweep_9
file added
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M19\M19_JS_A1_C02.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.19it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 3/3 [00:01<00:00,  1.75it/s]


Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_5
Sweep_6
Sweep_7
Sweep_8
Sweep_9
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M19\M19_JS_A1_C03.nwb


...

Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_55
Sweep_56
Sweep_59
Sweep_7
Sweep_8
Sweep_9
file added
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M19\M19_SA_A1_C09.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M19\M19_SA_A1_C10.nwb


Spike: 100%|███████████████████████████████████████████████████████████████████████████| 13/13 [00:02<00:00,  4.50it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


Sweep_11
Sweep_12
Sweep_13
Sweep_14
file added


In [72]:
M19_df.tail()

,ramp_amp,inflection_time,inflection_amp,peak_amp,peak_width,peak_sharpness,exp_lambda,exp_const,isi,r_squared_ramp,r_squared_exp,Sweep_#,file_name,dendriticType,SomaLayerLoc,brainOrigin,Weight,Sex,Age,Species
8,1.006392,0.40,-32.806397,5.126953,0.40,6.530762,7.080046,-45.984471,179.90,0.909565,0.810322,Sweep_14,M19_SA_A1_C10,NA,NA,V1,14.6,M,14,Macaca fascicularis
9,1.422165,0.45,-30.914307,5.889893,0.40,5.783081,5.853469,-46.889182,56.20,0.907662,0.941953,Sweep_14,M19_SA_A1_C10,NA,NA,V1,14.6,M,14,Macaca fascicularis
10,1.713574,0.45,-29.846192,5.950928,0.40,5.493164,6.284264,-46.386262,263.10,0.955074,0.886901,Sweep_14,M19_SA_A1_C10,NA,NA,V1,14.6,M,14,Macaca fascicularis
11,1.909529,0.45,-30.212403,5.676270,0.35,5.187988,6.138267,-46.991332,138.65,0.989066,0.926262,Sweep_14,M19_SA_A1_C10,NA,NA,V1,14.6,M,14,Macaca fascicularis
12,1.524043,0.45,-30.059815,6.103516,0.40,5.844116,5.572046,-46.961211,NaN,0.885470,0.956144,Sweep_14,M19_SA_A1_C10,NA,NA,V1,14.6,M,14,Macaca fascicularis


In [73]:
file_paths10 = get_file_paths(r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M20")
M20_df = monkey_df(file_paths10, 174)

save_dataframe_to_pickle(M20_df, r"C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M20_df.pkl")

C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M20\M20_JS_A1_C01.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 19/19 [00:03<00:00,  5.92it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████| 151/151 [00:03<00:00, 39.40it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')


Sweep_13
Sweep_14
Sweep_22
Sweep_23
Sweep_24
Sweep_25
Sweep_26
file added
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M20\M20_JS_A1_C02.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 25/25 [00:03<00:00,  6.38it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.94s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.10s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.99s/

Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_15
Sweep_16
Sweep_50
Sweep_51
Sweep_52
Sweep_53
Sweep_54
Sweep_55
Sweep_56
Sweep_57
Sweep_58
Sweep_59
Sweep_60
Sweep_61
Sweep_62
Sweep_63
Sweep_64
Sweep_65
Sweep_66
Sweep_67
Sweep_68
Sweep_69
Sweep_70
Sweep_71
Sweep_72
Sweep_73
Sweep_74
Sweep_75
Sweep_76
Sweep_77
Sweep_78
Sweep_79
Sweep_80
Sweep_81
Sweep_82
Sweep_83
Sweep_84
Sweep_85
file added
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M20\M20_JS_A1_C03.nwb


...

Sweep_50
Sweep_52
Sweep_6
Sweep_7
Sweep_8
file added
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M20\M20_SA_A1_C08.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 8/8 [00:02<00:00,  2.99it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.84s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.32s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 7/7 [00:02<00:00,  3.09it

Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_27
Sweep_30
Sweep_31
Sweep_5
Sweep_6
Sweep_7
Sweep_8
Sweep_9
file added
C:\Users\david\Documents\Voytek Research\nwb files\primate Dataset\M20\M20_SA_A1_C09.nwb


C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 35/35 [00:05<00:00,  6.30it/s]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|█████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.29s/it]
C:\Users\david\spikeparam\spikeparam\patch\fit\fit.py:228: UserWarning: No spikes detected.
  warnings.warn('No spikes detected.')
Spike: 100%|███████████████████████████████████████████████████████████████████████████| 37/37 [00:04<00:00,  7.74it/s]


Sweep_10
Sweep_11
Sweep_12
Sweep_13
Sweep_14
Sweep_15
Sweep_16
Sweep_17
Sweep_18
Sweep_19
Sweep_72
Sweep_73
Sweep_9
file added
